In [2]:
import os
import xml.etree.ElementTree as ET
import pandas as pd

from dataset import data_dir

In [44]:
def parse_tasks(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    # Navigate to the tasks node
    tasks = {}
    for task in root.findall(".//task"):
        task_data = {
            "name": task.find("name").text,
            "size": int(task.find("size").text),
            "start_frame": int(task.find("start_frame").text),
            "stop_frame": int(task.find("stop_frame").text),
            "resolution": {
                "width": int(task.find("original_size/width").text),
                "height": int(task.find("original_size/height").text)
            }
        }
        tasks[int(task.find("id").text)] = task_data

    return tasks

def parse_tracks(match_dir_path):
    for xml_filename in os.listdir(os.path.join(match_dir_path, "annotations")):
        xml_file = os.path.join(match_dir_path, "annotations", xml_filename)
        tree = ET.parse(xml_file)
        root = tree.getroot()

        task_data = parse_tasks(xml_file)
        
        last_start_frame_actual = 0
        last_size = 0
        for task_id in task_data:
            if last_size == 0:
                task_data[task_id]['start_frame_actual'] = 0
                last_start_frame_actual = task_data[task_id]['start_frame_actual']
                last_size = task_data[task_id]['size']
            else:
                task_data[task_id]['start_frame_actual'] = last_start_frame_actual + last_size
                last_start_frame_actual = task_data[task_id]['start_frame_actual']
                last_size = task_data[task_id]['size']

        for track in root.findall(".//track"):
            track_info = track.attrib
            #print(track_info)
            frames = []
            start_frame = None
            tracking_outside = 0
            for point in track.iter('points'):
                point_info = point.attrib

                # init the start_frame of the track
                if start_frame == None:
                    start_frame = int(task_data[int(track_info['task_id'])]['start_frame_actual'])
                    # If the calculated start frame (actual) does not equal the first point, then the clip starts with the ball outside
                    if start_frame != int(point_info['frame']):
                        tracking_outside = 1
                        outside_start_frame = 0

                if point_info['outside'] == '1':
                    if tracking_outside == 0:
                        tracking_outside = 1
                        outside_start_frame = int(point_info['frame'])-start_frame   
                else:
                    # Add in all the frames the ball was not visible for
                    if tracking_outside == 1:
                        #print(outside_start_frame, int(point_info['frame'])-start_frame)
                        #print(int(point_info['frame']), start_frame)
                        for i in range(outside_start_frame, int(point_info['frame'])-start_frame):
                            #print(i, 0, 0.0, 0.0)
                            frames.append({
                                "Frame":i,
                                "Visibility":0, # flipping 0's and 1's
                                "X":0.0,
                                "Y":0.0
                            })
                        tracking_outside = 0
                        outside_start_frame = 0
                    #print(int(point_info['frame'])-start_frame, 1, float(point_info['points'].split(',')[0]), float(point_info['points'].split(',')[1]))
                    frames.append({
                        "Frame":int(point_info['frame'])-start_frame,
                        "Visibility":1, # flipping 0's and 1's
                        "X":float(point_info['points'].split(',')[0]),
                        "Y":float(point_info['points'].split(',')[1])
                    })
            
            # If ends on an outside point
            if tracking_outside == 1:
                #print(outside_start_frame, int(point_info['frame'])-start_frame)
                #print(int(point_info['frame']), start_frame)
                for i in range(outside_start_frame, task_data[int(track_info['task_id'])]['size']):
                    #print(i, 0, 0.0, 0.0)
                    frames.append({
                        "Frame":i,
                        "Visibility":0, # flipping 0's and 1's
                        "X":0.0,
                        "Y":0.0
                    })
                tracking_outside = 0
                outside_start_frame = 0
                    

            name, ext = task_data[int(track.attrib['task_id'])]['name'].rsplit('.', 1)
            file_name = f"{name}_ball.csv"
            
            csv_dir_path = os.path.join(match_dir_path, 'csv')
            os.makedirs(csv_dir_path, exist_ok=True)
            
            df = pd.DataFrame(frames)
            df.to_csv(os.path.join(csv_dir_path, file_name), index=False)

In [45]:
for match_dir in os.listdir(os.path.join(data_dir, "train")):
    match_dir_path = os.path.join(data_dir, "train", match_dir)
    parse_tracks(match_dir_path)

In [37]:
match_dir_path = os.path.join(data_dir, "train", "match1")
parse_tracks(match_dir_path)




{'id': '0', 'label': 'ball', 'source': 'manual', 'task_id': '1302083', 'subset': 'default'}
0 2
2 0
0 0 0.0 0.0
1 0 0.0 0.0
2 1 553.8 457.79
3 1 557.78 456.47
4 1 561.75 455.15
5 1 565.73 453.82
6 1 569.7 452.5
7 1 572.3 451.6
8 1 575.7 453.6
9 1 579.1 457.45
10 1 582.5 461.3
11 1 586.05 468.8
12 1 589.6 476.3
13 1 610.0 459.9
14 1 643.5 421.0
15 1 669.8 392.3
16 1 691.6 369.6
17 1 710.4 352.4
18 1 724.8 339.5
19 1 738.8 328.7
20 1 749.7 321.1
21 1 759.9 314.6
22 1 769.2 310.6
23 1 777.3 308.2
24 1 784.3 307.35
25 1 791.3 306.5
26 1 797.15 307.5
27 1 803.0 308.5
28 1 807.5 310.9
29 1 812.0 313.3
30 1 816.6 316.1
31 1 820.2 320.07
32 1 823.8 324.03
33 1 827.4 328.0
34 1 830.45 333.0
35 1 833.5 338.0
36 1 836.2 343.55
37 1 838.9 349.1
38 1 841.3 354.83
39 1 843.7 360.57
40 1 846.1 366.3
41 42
42 0
41 0 0.0 0.0
42 1 850.3 379.8
43 1 852.1 386.3
44 1 853.9 392.8
45 1 855.1 387.83
46 1 856.3 382.87
47 1 857.5 377.9
48 49
49 0
48 0 0.0 0.0
49 1 859.8 370.0
50 1 860.88 368.18
51 1 861.95 366.

In [35]:
match_dir_path = os.path.join(data_dir, "train", "match1")
task_data = parse_tracks(match_dir_path)
task_data



0 216
216 338
554 201
755 202


{1302083: {'name': 'rally_1.mp4',
  'size': 216,
  'start_frame': 0,
  'stop_frame': 215,
  'resolution': {'width': 1280, 'height': 720},
  'start_frame_actual': 0},
 1302084: {'name': 'rally_2.mp4',
  'size': 338,
  'start_frame': 0,
  'stop_frame': 337,
  'resolution': {'width': 1280, 'height': 720},
  'start_frame_actual': 216},
 1302085: {'name': 'rally_3.mp4',
  'size': 201,
  'start_frame': 0,
  'stop_frame': 200,
  'resolution': {'width': 1280, 'height': 720},
  'start_frame_actual': 554},
 1302086: {'name': 'rally_4.mp4',
  'size': 202,
  'start_frame': 0,
  'stop_frame': 201,
  'resolution': {'width': 1280, 'height': 720},
  'start_frame_actual': 755},
 1302087: {'name': 'rally_5.mp4',
  'size': 307,
  'start_frame': 0,
  'stop_frame': 306,
  'resolution': {'width': 1280, 'height': 720},
  'start_frame_actual': 957}}